In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import json
import os
from tqdm import tqdm

# --- 1. 定义所有路径 ---

# (输入) 包含所有价格文件的目录 (ETH 计价)
PRICE_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\every_icon_price_sequence_in_eth")

# (输入) 包含模拟结果的基础目录
DATA_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data")

# (输入) 模拟日志 CSV (我们的主循环文件)
LOG_FILE = DATA_DIR / "simulation_log.csv"

# (输入) HF 波动 CSV 文件夹
HF_DIR = DATA_DIR / "HF_fluctuation_for_samples" / "HF_fluctuation"

# (输入) HF 配方 JSON 文件夹
RECIPE_DIR = DATA_DIR / "HF_fluctuation_for_samples" / "HF_sample_recipe"

# (输出) 存储图表的新基础目录
VISUAL_OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\visual")

# 确保输出目录存在
VISUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"图表输出目录已准备好: {VISUAL_OUTPUT_DIR}")

# --- 2. 优化：预加载所有价格数据 ---

def load_all_price_data(price_dir):
    """
    (优化) 预先加载所有价格文件到内存中。
    返回: {'AAVE': DataFrame, 'USDC': DataFrame, ...}
    """
    price_data_cache = {}
    print(f"\n--- 正在预加载所有价格数据(ETH计价) ---")
    
    price_files = [f for f in os.listdir(price_dir) if f.endswith('.csv')]
    
    for filename in tqdm(price_files, desc="预加载价格文件"):
        symbol = filename.split('_')[0]
        file_path = price_dir / filename
        try:
            df = pd.read_csv(file_path)
            if df.empty:
                continue
            # 解析时间并设置为索引，以便快速查找
            df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], format='ISO8601')
            df = df.set_index('datetime_utc')
            price_data_cache[symbol] = df
        except Exception as e:
            print(f"警告: 加载 {filename} 失败: {e}")
            
    print(f"--- 成功预加载 {len(price_data_cache)} 个价格序列 ---")
    return price_data_cache

# 执行预加载
price_data_cache = load_all_price_data(PRICE_DIR)

# --- 3. 加载主日志文件 ---
try:
    # (!! 修复 Windows MAX_PATH 路径过长错误 !!)
    log_file_long_path = f"\\\\?\\{LOG_FILE.resolve()}"
    df_log = pd.read_csv(log_file_long_path)
except FileNotFoundError:
    print(f"严重错误: 找不到日志文件 {LOG_FILE}。无法继续。")
    raise

# 筛选出我们成功处理的样本
df_success = df_log[df_log['status'] == 'Success'].copy()
print(f"\n--- 找到 {len(df_success)} 个成功模拟的样本。开始生成图表... ---")

# --- 4. 主循环：为每个样本生成图表 ---

for index, row in tqdm(df_success.iterrows(), total=df_success.shape[0], desc="生成所有样本图表"):
    
    tx_hash = row['txHash']
    hf_csv_name = row['output_csv_file']
    recipe_json_name = row['output_recipe_json_file']
    
    # 1. 创建该样本的输出文件夹 (使用长路径修复)
    sample_output_folder = VISUAL_OUTPUT_DIR / tx_hash
    long_folder_path = f"\\\\?\\{sample_output_folder.resolve()}"
    os.makedirs(long_folder_path, exist_ok=True)
    
    try:
        # --- 2. 加载该样本的数据 ---
        
        # a. 加载 HF 波动 CSV (使用长路径修复)
        df_hf_path = HF_DIR / hf_csv_name
        df_hf_path_long = f"\\\\?\\{df_hf_path.resolve()}"
        df_hf = pd.read_csv(df_hf_path_long)
        df_hf['datetime_utc'] = pd.to_datetime(df_hf['datetime_utc'])
        
        # b. 加载 "配方" JSON (使用长路径修复)
        recipe_path = RECIPE_DIR / recipe_json_name
        recipe_path_long = f"\\\\?\\{recipe_path.resolve()}"
        with open(recipe_path_long, 'r', encoding='utf-8') as f:
            recipe = json.load(f)
            
        # c. 从配方中提取所需信息
        symbols_needed = set()
        for asset in recipe['numerator_collaterals']:
            symbols_needed.add(asset['symbol'])
        for asset in recipe['denominator_debts']:
            symbols_needed.add(asset['symbol'])
        
        hf_true_onchain = recipe['health_factor_anchors']['HF_True_OnChain']

        # --- 3. 绘制图表 1 (健康因子) ---
        plt.figure(figsize=(14, 7))
        ax1 = sns.lineplot(
            data=df_hf, 
            x='datetime_utc', 
            y='health_factor', 
            label='Simulated Health Factor'
        )
        
        ax1.axhline(
            1.0, 
            color='red', 
            linestyle='--', 
            label='Liquidation Threshold (HF=1.0)'
        )
        ax1.axhline(
            hf_true_onchain, 
            color='darkorange', 
            linestyle=':', 
            label=f'Actual On-Chain HF ({hf_true_onchain:.4f})'
        )
        
        ax1.set_title(f'Health Factor Fluctuation (Adjusted)\nSample: {tx_hash}', fontsize=16)
        ax1.set_xlabel('Date (UTC)', fontsize=12)
        ax1.set_ylabel('Health Factor', fontsize=12)
        ax1.legend()
        ax1.grid(True)
        
        # (使用长路径修复)
        hf_plot_path = sample_output_folder / "health_factor_fluctuation.png"
        hf_plot_path_long = f"\\\\?\\{hf_plot_path.resolve()}"
        plt.tight_layout()
        plt.savefig(hf_plot_path_long)
        plt.close() # 关闭图表以释放内存

        # --- 4. 绘制图表 2 (资产价格) ---
        
        # a. 从缓存中准备价格数据
        dfs_to_plot = []
        for symbol in symbols_needed:
            if symbol in price_data_cache:
                dfs_to_plot.append(price_data_cache[symbol][['price_in_eth']].rename(columns={'price_in_eth': symbol}))
            else:
                tqdm.write(f"  - 警告 ({tx_hash}): 找不到 {symbol} 的预加载价格数据。")

        if not dfs_to_plot:
            tqdm.write(f"  - 警告 ({tx_hash}): 没有找到任何资产的价格数据。跳过价格图。")
            continue

        # b. 合并并筛选时间
        df_prices_wide = pd.concat(dfs_to_plot, axis=1)
        
        time_min = df_hf['datetime_utc'].min()
        time_max = df_hf['datetime_utc'].max()
        df_prices_filtered = df_prices_wide.loc[time_min:time_max]

        # c. 转换为长格式以便 Seaborn 使用
        df_prices_long = df_prices_filtered.melt(
            var_name='Asset', 
            value_name='Price (ETH)', 
            ignore_index=False
        ).reset_index()

        # e. (!! 已更正: 使用 relplot 绘制子图 !!)
        if df_prices_long.empty:
            tqdm.write(f"  - 警告 ({tx_hash}): 筛选时间后 df_prices_long 为空。跳过价格图。")
            continue

        num_assets = len(symbols_needed)
        col_wrap_num = min(3, num_assets) # 每行最多3个图

        # --- (!! 关键修复 !!) ---
        # `sharey` 和 `sharex` 必须在 `facet_kws` 字典中传递
        g = sns.relplot(
            data=df_prices_long,
            x='datetime_utc',
            y='Price (ETH)',
            col='Asset',         # 按资产分列
            hue='Asset',         # 每个图用自己的颜色 (与 col 匹配)
            kind='line',
            col_wrap=col_wrap_num, # 每行最多3个图
            height=4,            # 每个子图的高度
            aspect=1.5,          # 每个子图的宽高比
            legend=False,        # 关闭默认图例 (因为标题已说明一切)
            facet_kws={
                'sharey': False, # <-- (!! 已修复 !!) 独立的 Y 轴
                'sharex': True   # <-- (!! 已修复 !!) 共享的 X 轴
            }
        )
        # --- (修复结束) ---
        
        g.fig.suptitle(f'Involved Asset Prices (ETH Denominated)\nSample: {tx_hash}', fontsize=16, y=1.03)
        g.set_axis_labels("Date (UTC)", "Price (ETH)")
        g.set_titles("{col_name}") # 设置每个子图的标题 (例如 "WETH", "USDC")
        g.tight_layout(rect=[0, 0, 1, 0.95]) # 调整布局为总标题腾出空间
        
        # f. 保存图表 2 (使用长路径修复)
        price_plot_path = sample_output_folder / "asset_price_fluctuation.png"
        price_plot_path_long = f"\\\\?\\{price_plot_path.resolve()}"
        plt.savefig(price_plot_path_long)
        plt.close() # 关闭图表

    except Exception as e:
        tqdm.write(f"!! 处理 {tx_hash} 时发生严重错误: {e}")

print(f"\n--- 可视化全部完成 ---")
print(f"所有图表均已保存到 {VISUAL_OUTPUT_DIR} 下的各个子文件夹中。")

图表输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\visual

--- 正在预加载所有价格数据(ETH计价) ---


预加载价格文件: 100%|██████████| 10/10 [00:00<00:00, 130.98it/s]


--- 成功预加载 10 个价格序列 ---

--- 找到 16 个成功模拟的样本。开始生成图表... ---


生成所有样本图表:   0%|          | 0/16 [00:00<?, ?it/s]c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
生成所有样本图表:   6%|▋         | 1/16 [00:01<00:21,  1.46s/it]c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
生成所有样本图表:  12%|█▎        | 2/16 [00:02<00:18,  1.31s/it]c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._fi


--- 可视化全部完成 ---
所有图表均已保存到 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\visual 下的各个子文件夹中。


更改坐标重合以及边框问题

In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import json
import os
from tqdm import tqdm

# --- 1. 定义所有路径 ---

# (输入) 包含所有价格文件的目录 (ETH 计价)
PRICE_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\every_icon_price_sequence_in_eth")

# (输入) 包含模拟结果的基础目录
DATA_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data")

# (输入) 模拟日志 CSV (我们的主循环文件)
LOG_FILE = DATA_DIR / "simulation_log.csv"

# (输入) HF 波动 CSV 文件夹
HF_DIR = DATA_DIR / "HF_fluctuation_for_samples" / "HF_fluctuation"

# (输入) HF 配方 JSON 文件夹
RECIPE_DIR = DATA_DIR / "HF_fluctuation_for_samples" / "HF_sample_recipe"

# (输出) 存储图表的新基础目录
VISUAL_OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\visual")

# 确保输出目录存在
VISUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"图表输出目录已准备好: {VISUAL_OUTPUT_DIR}")

# --- 2. 优化：预加载所有价格数据 ---

def load_all_price_data(price_dir):
    """
    (优化) 预先加载所有价格文件到内存中。
    返回: {'AAVE': DataFrame, 'USDC': DataFrame, ...}
    """
    price_data_cache = {}
    print(f"\n--- 正在预加载所有价格数据(ETH计价) ---")
    
    price_files = [f for f in os.listdir(price_dir) if f.endswith('.csv')]
    
    for filename in tqdm(price_files, desc="预加载价格文件"):
        symbol = filename.split('_')[0]
        file_path = price_dir / filename
        try:
            df = pd.read_csv(file_path)
            if df.empty:
                continue
            # 解析时间并设置为索引，以便快速查找
            df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], format='ISO8601')
            df = df.set_index('datetime_utc')
            price_data_cache[symbol] = df
        except Exception as e:
            print(f"警告: 加载 {filename} 失败: {e}")
            
    print(f"--- 成功预加载 {len(price_data_cache)} 个价格序列 ---")
    return price_data_cache

# 执行预加载
price_data_cache = load_all_price_data(PRICE_DIR)

# --- 3. 加载主日志文件 ---
try:
    # (修复 Windows MAX_PATH 路径过长错误)
    log_file_long_path = f"\\\\?\\{LOG_FILE.resolve()}"
    df_log = pd.read_csv(log_file_long_path)
except FileNotFoundError:
    print(f"严重错误: 找不到日志文件 {LOG_FILE}。无法继续。")
    raise

# 筛选出我们成功处理的样本
df_success = df_log[df_log['status'] == 'Success'].copy()
print(f"\n--- 找到 {len(df_success)} 个成功模拟的样本。开始生成图表... ---")

# --- 4. 主循环：为每个样本生成图表 ---

for index, row in tqdm(df_success.iterrows(), total=df_success.shape[0], desc="生成所有样本图表"):
    
    tx_hash = row['txHash']
    hf_csv_name = row['output_csv_file']
    recipe_json_name = row['output_recipe_json_file']
    
    # 1. 创建该样本的输出文件夹 (使用长路径修复)
    sample_output_folder = VISUAL_OUTPUT_DIR / tx_hash
    long_folder_path = f"\\\\?\\{sample_output_folder.resolve()}"
    os.makedirs(long_folder_path, exist_ok=True)
    
    try:
        # --- 2. 加载该样本的数据 ---
        
        # a. 加载 HF 波动 CSV (使用长路径修复)
        df_hf_path = HF_DIR / hf_csv_name
        df_hf_path_long = f"\\\\?\\{df_hf_path.resolve()}"
        df_hf = pd.read_csv(df_hf_path_long)
        df_hf['datetime_utc'] = pd.to_datetime(df_hf['datetime_utc'])
        
        # b. 加载 "配方" JSON (使用长路径修复)
        recipe_path = RECIPE_DIR / recipe_json_name
        recipe_path_long = f"\\\\?\\{recipe_path.resolve()}"
        with open(recipe_path_long, 'r', encoding='utf-8') as f:
            recipe = json.load(f)
            
        # c. 从配方中提取所需信息
        symbols_needed = set()
        for asset in recipe['numerator_collaterals']:
            symbols_needed.add(asset['symbol'])
        for asset in recipe['denominator_debts']:
            symbols_needed.add(asset['symbol'])
        
        hf_true_onchain = recipe['health_factor_anchors']['HF_True_OnChain']

        # --- 3. 绘制图表 1 (健康因子) ---
        plt.figure(figsize=(14, 7))
        ax1 = sns.lineplot(
            data=df_hf, 
            x='datetime_utc', 
            y='health_factor', 
            label='Simulated Health Factor'
        )
        
        ax1.axhline(
            1.0, 
            color='red', 
            linestyle='--', 
            label='Liquidation Threshold (HF=1.0)'
        )
        ax1.axhline(
            hf_true_onchain, 
            color='darkorange', 
            linestyle=':', 
            label=f'Actual On-Chain HF ({hf_true_onchain:.4f})'
        )
        
        ax1.set_title(f'Health Factor Fluctuation (Adjusted)\nSample: {tx_hash}', fontsize=16)
        ax1.set_xlabel('Date (UTC)', fontsize=12)
        ax1.set_ylabel('Health Factor', fontsize=12)
        ax1.legend()
        ax1.grid(True)
        
        # (使用长路径修复)
        hf_plot_path = sample_output_folder / "health_factor_fluctuation.png"
        hf_plot_path_long = f"\\\\?\\{hf_plot_path.resolve()}"
        plt.tight_layout()
        plt.savefig(hf_plot_path_long)
        plt.close() # 关闭图表以释放内存

        # --- 4. 绘制图表 2 (资产价格) ---
        
        # a. 从缓存中准备价格数据
        dfs_to_plot = []
        for symbol in symbols_needed:
            if symbol in price_data_cache:
                dfs_to_plot.append(price_data_cache[symbol][['price_in_eth']].rename(columns={'price_in_eth': symbol}))
            else:
                tqdm.write(f"  - 警告 ({tx_hash}): 找不到 {symbol} 的预加载价格数据。")

        if not dfs_to_plot:
            tqdm.write(f"  - 警告 ({tx_hash}): 没有找到任何资产的价格数据。跳过价格图。")
            continue

        # b. 合并并筛选时间
        df_prices_wide = pd.concat(dfs_to_plot, axis=1)
        
        time_min = df_hf['datetime_utc'].min()
        time_max = df_hf['datetime_utc'].max()
        df_prices_filtered = df_prices_wide.loc[time_min:time_max]

        # c. 转换为长格式以便 Seaborn 使用
        df_prices_long = df_prices_filtered.melt(
            var_name='Asset', 
            value_name='Price (ETH)', 
            ignore_index=False
        ).reset_index()

        # e. (!! 已更正: 使用 relplot 绘制子图 !!)
        if df_prices_long.empty:
            tqdm.write(f"  - 警告 ({tx_hash}): 筛选时间后 df_prices_long 为空。跳过价格图。")
            continue

        num_assets = len(symbols_needed)
        col_wrap_num = min(3, num_assets) # 每行最多3个图

        g = sns.relplot(
            data=df_prices_long,
            x='datetime_utc',
            y='Price (ETH)',
            col='Asset',         
            hue='Asset',         
            kind='line',
            col_wrap=col_wrap_num, 
            height=4,            
            aspect=1.5,          
            legend=False,        
            facet_kws={
                'sharey': False, # 独立的 Y 轴
                'sharex': True   # 共享的 X 轴
            }
        )
        
        # --- (!! 关键修复 !!) ---
        
        # 1. (修复标题) 设置总标题, y=1.03 稍微调高以避免重叠
        g.fig.suptitle(f'Involved Asset Prices (ETH Denominated)\nSample: {tx_hash}', fontsize=16, y=1.03)
        
        # 2. (修复坐标轴) 设置轴标签
        g.set_axis_labels("Date (UTC)", "Price (ETH)")
        
        # 3. (修复坐标轴) 设置子图标题
        g.set_titles("{col_name}") 
        
        # 4. (修复X轴重叠) 旋转 X 轴刻度标签
        g.set_xticklabels(rotation=30, ha='right')
        
        # 5. (修复边框/标题被剪裁) 调整布局
        g.tight_layout(rect=[0, 0, 1, 0.98]) # 为总标题留出空间
        
        # 6. (!! 核心修复 !!) 使用 g.savefig() 代替 plt.savefig()
        price_plot_path = sample_output_folder / "asset_price_fluctuation.png"
        price_plot_path_long = f"\\\\?\\{price_plot_path.resolve()}"
        g.savefig(price_plot_path_long) 
        plt.close() # 关闭图表
        
        # --- (修复结束) ---

    except Exception as e:
        tqdm.write(f"!! 处理 {tx_hash} 时发生严重错误: {e}")

print(f"\n--- 可视化全部完成 ---")
print(f"所有图表均已保存到 {VISUAL_OUTPUT_DIR} 下的各个子文件夹中。")

图表输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\visual

--- 正在预加载所有价格数据(ETH计价) ---


预加载价格文件: 100%|██████████| 10/10 [00:00<00:00, 131.82it/s]


--- 成功预加载 10 个价格序列 ---

--- 找到 16 个成功模拟的样本。开始生成图表... ---


生成所有样本图表:   0%|          | 0/16 [00:00<?, ?it/s]c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
生成所有样本图表:   6%|▋         | 1/16 [00:01<00:20,  1.35s/it]c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
生成所有样本图表:  12%|█▎        | 2/16 [00:02<00:17,  1.27s/it]c:\Users\10158\anaconda3\envs\aave\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._fi


--- 可视化全部完成 ---
所有图表均已保存到 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\visual 下的各个子文件夹中。
